# 03 · Pincode areas (CPU)

**What this does:** works out which pincode every building belongs to.

**Why:** several benchmark features depend on it: writing the pincode into an address, the "wrong pincode" noise, and the *misroute* metric (did the geocoder land in the wrong pincode area?).

**Two ways, best one wins**

| Way | How | Quality |
|---|---|---|
| A. Boundary shapes | DataMeet's copy of India Post's Hyderabad pincode shapes (2013) | exact where the file is still right; misses newer pincodes |
| B. Nearest labelled points | OpenStreetMap and Overture places that carry a pincode vote for their surroundings (k-nearest-neighbours, kNN). Overture's business-listed pincodes are noisy, so they are cleaned first | about 9 in 10 correct (measured below) |

The notebook tries both, grades them on the same answer key, and keeps the more accurate one.

**Runtime:** a few minutes, CPU. **Needs:** notebook 01 (and 02 if you ran it).

In [ ]:
# Setup (the same cell opens every notebook): fetch the project code, install what is missing, find the data folder.
import os, sys, subprocess
REPO = 'SiddharthaRaja/Navigate'
PIP  = ['h3', 'rapidfuzz']
on_cloud = os.path.exists('/content') or os.path.exists('/kaggle/working')
if on_cloud:
    CODE = '/kaggle/working/Navigate-code' if os.path.exists('/kaggle/working') else '/content/Navigate'
    if not os.path.exists(f'{CODE}/bench'):
        token = os.environ.get('GH_TOKEN', '')
        try:                                   # only needed while the GitHub repo is private
            from google.colab import userdata; token = token or userdata.get('GH_TOKEN')
        except Exception: pass
        try:
            from kaggle_secrets import UserSecretsClient; token = token or UserSecretsClient().get_secret('GH_TOKEN')
        except Exception: pass
        url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
        r = subprocess.run(['git', 'clone', '--depth', '1', url, CODE], env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
        assert r.returncode == 0, 'Could not fetch the code. If the repo is private, add a GitHub token as the secret GH_TOKEN.'
    else:
        subprocess.run(['git', '-C', CODE, 'pull', '--ff-only'])
    if PIP: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PIP], check=True)
else:
    CODE = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
sys.path.insert(0, CODE)
import numpy as np, pandas as pd
from bench import env
P = env.setup('hyderabad')
print('running on', P['where'], '| data folder:', P['root'])

## 1. Way B first: pincode from nearby labelled points
This always works, and it gives us something to check the boundary file against.

In [ ]:
from bench import pincode as pc
pts = pc.labelled_points(P['gaz'])      # Overture's noisy pincodes are cleaned inside this step
print(f"{len(pts):,} points with a pincode after cleaning ({pts.src.value_counts().to_dict()}), {pts.pin.nunique()} different pincodes")

knn = pc.knn_lookup(pts)
# Accuracy check: hide each OpenStreetMap point in turn and ask its neighbours for its pincode.
# Only OSM points are used as the answer key: they were typed by mappers and are cleaner than business listings.
osm = pts[pts.src == 'osm']
guess, conf = knn(osm.lat.values, osm.lon.values, exclude_self=True)
loo = float((guess == osm.pin.values).mean())
print(f"accuracy of the neighbour vote on {len(osm):,} OpenStreetMap points: {loo:.1%}")

## 2. Way A: official boundary shapes
If the download fails (the government portal is often down), the notebook carries on with Way B. You can also drop a file at `raw/pincode_boundaries.geojson` by hand and re-run.

In [ ]:
import geopandas as gpd, shapely, urllib.request
BBOX = (78.25, 17.25, 78.70, 17.60)
# DataMeet's copy of India Post's Hyderabad pincode shapes (2013, CC BY-SA 2.5 India). Old: pincodes created since are missing,
# so it is only used if it beats the neighbour vote below. The government's own portal is often down.
SOURCES = ['https://raw.githubusercontent.com/datameet/PincodeBoundary/master/Hyderabad/boundary.geojson']
raw_path = f"{P['raw']}/pincode_boundaries.geojson"
if os.path.exists(raw_path) and b'"Feature' not in open(raw_path, 'rb').read(5000):
    os.remove(raw_path); print('removed a broken download from an earlier run')
if not os.path.exists(raw_path):
    for url in SOURCES:
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            data = urllib.request.urlopen(req, timeout=120).read()
            if b'"Feature' not in data[:5000]:            # a web page instead of map data: the file has moved
                print('not a GeoJSON file, skipped:', url); continue
            open(raw_path, 'wb').write(data)
            print('downloaded', url); break
        except Exception as e:
            print('could not download', url, '->', e)

areas = None
if os.path.exists(raw_path):
    try:
        g = gpd.read_file(raw_path, bbox=BBOX)
        col, share = pc.find_pin_column(g)
        assert col is not None, f'no pincode column found (best match {share:.0%})'
        g = g.rename(columns={col: 'pin'})[['pin', 'geometry']]
        g['pin'] = g['pin'].astype(str).str.replace(r'\.0$', '', regex=True)
        g['geometry'] = g.geometry.make_valid()
        areas = g[g.intersects(shapely.box(*BBOX))].dissolve('pin').reset_index()
        print(f"{len(areas)} pincode shapes cover the study area (pincode column was '{col}')")
    except Exception as e:
        print('boundary file unusable ->', e)

## 3. Choose
Both methods are graded on the same OpenStreetMap points. The boundary file is used only if it covers almost everything and is more accurate than the neighbour vote.

In [ ]:
method, agree = 'knn', None
if areas is not None and len(areas) >= 20:
    poly = pc.polygon_lookup(areas)
    got, _ = poly(osm.lat.values, osm.lon.values)
    got = np.asarray(got, object)
    covered = float((got != None).mean())
    agree = float((got == osm.pin.values)[got != None].mean())
    print(f"boundary file: covers {covered:.1%} of the OSM points, right {agree:.1%} of the time | neighbour vote: right {loo:.1%}")
    if covered >= 0.95 and agree > loo:
        method = 'polygons'
        areas.to_parquet(f"{P['gaz']}/pincode_areas.parquet", index=False)
if method == 'knn' and os.path.exists(f"{P['gaz']}/pincode_areas.parquet"):
    os.remove(f"{P['gaz']}/pincode_areas.parquet")
print('USING:', 'boundary file' if method == 'polygons' else 'neighbour vote (kNN)')

import json
json.dump({'method': method, 'knn_leave_one_out_accuracy': round(loo, 4), 'polygon_agreement': agree,
           'labelled_points': len(pts), 'pincodes': int(pts.pin.nunique())}, open(f"{P['gaz']}/pincode_method.json", 'w'), indent=2)

## 4. Check on a map
Each dot is a labelled point, coloured by pincode. Neighbouring dots should share a colour; a salt-and-pepper pattern means the labels are noisy.

In [ ]:
import matplotlib.pyplot as plt
codes = pts.pin.astype('category').cat.codes
plt.figure(figsize=(8, 7)); plt.scatter(pts.lon, pts.lat, c=codes % 20, cmap='tab20', s=4)
plt.gca().set_aspect('equal'); plt.title(f'{pts.pin.nunique()} pincodes from labelled points'); plt.show()

## Share the outputs
Packs only the files this notebook created into one zip in `My Drive > Navigate > data > share`. Download that one file to share or back up the results.

In [ ]:
env.pack_outputs(P, '03' + '_outputs', [f"{P['gaz']}/pincode_method.json", f"{P['gaz']}/pincode_areas.parquet"])